# Silicate Weathering and Carbon Isotope Dynamics in the Cenozoic
### EMSC2010 — Individual Data Analysis Project (Assessment 3)

| Field | Detail |
|---|---|
| **Student ID** | u8163685 |
| **Course** | EMSC2010 — Data Analysis in Earth Sciences, ANU |
| **Repository** | https://github.com/Sameer15200/EMSC2010InduvidualProjectu8163685 |

## How to Run
1. Run **Cell 1** first — installs PyMC and ArviZ (~2 min)
2. Run all cells **in order, top to bottom**
3. Four MCMC sampling cells — each takes ~2–4 minutes
4. All figures save to `figures/` automatically

> No manual downloads required. CENOGRID loads from PANGAEA. Sr data is embedded.

---

## 1. Project Overview

### 1.1 Scientific Background

Earth's long-term climate is regulated by the **silicate weathering thermostat** — a negative feedback connecting atmospheric CO₂ to continental weathering:

$$\text{CaSiO}_3 + \text{CO}_2 \rightarrow \text{CaCO}_3 + \text{SiO}_2$$

Two geochemical proxies preserved in marine sediments allow us to track this system:

| Proxy | What it records | Cenozoic trend |
|-------|----------------|----------------|
| **⁸⁷Sr/⁸⁶Sr** | Balance between continental weathering and mantle hydrothermal input | Monotonically rising (0–66 Ma) |
| **δ¹³C** | Global carbon cycle: organic burial vs volcanic outgassing | Peaked ~15 Ma, then declined |

**The key scientific complexity:** Sr rose monotonically throughout the Cenozoic while δ¹³C peaked at the Mid-Miocene Carbon Optimum (~15 Ma) and declined since. This means the full Cenozoic record is unlikely to show a simple positive linear relationship — published literature confirms this (Caves et al. 2016; McArthur et al. 2020). However, during the **Oligocene–Mid-Miocene (34–15 Ma)**, both proxies were rising together, driven by Himalayan uplift and expanding ice sheets promoting organic carbon burial. A positive coupling is theoretically expected within this specific interval.

### 1.2 Research Question

> **How does the statistical relationship between seawater ⁸⁷Sr/⁸⁶Sr and benthic δ¹³C vary (a) across the full Cenozoic, (b) with the PETM excursion removed, and (c) within the Oligocene–Mid-Miocene interval (34–15 Ma)? What does a multi-interval Bayesian analysis reveal about the time-dependence of silicate weathering–carbon cycle coupling?**

### 1.3 Analytical Strategy — Three Bayesian Models

This project runs **three complete Bayesian linear regression models** targeting the same question at different temporal scales:

| Model | Dataset | Scientific Purpose |
|-------|---------|-------------------|
| **Model 1** | Full Cenozoic (0–66 Ma, n=67) | Establishes the long-term relationship |
| **Model 2** | PETM-exempt (0–66 Ma, 54–58 Ma removed, n=62) | Tests robustness to the largest outlier |
| **Model 3** | Oligocene–Mid-Miocene (34–15 Ma, n=20) | Tests the interval of expected positive coupling |

A **posterior comparison figure** then plots all three β posteriors on the same axis — directly visualising how the relationship evolves across Cenozoic time intervals.

### 1.4 Why Bayesian Analysis?

$$\underbrace{P(\theta \mid \text{data})}_{\text{posterior}} \propto \underbrace{P(\text{data} \mid \theta)}_{\text{likelihood}} \times \underbrace{P(\theta)}_{\text{prior}}$$

Bayesian methods are essential here because:
1. **Full uncertainty quantification** — each model produces a probability distribution over β, not a single number
2. **Comparable posteriors** — all three models use identical priors, making their posteriors directly comparable
3. **Formal model comparison** — WAIC tests whether Sr genuinely improves δ¹³C predictions
4. **Small sample handling** — Bayesian inference handles the n=20 sub-interval dataset more robustly than classical statistics

### 1.5 Datasets
- **δ¹³C:** Westerhold et al. (2020) CENOGRID — *Science* 369:1383 | doi:10.1594/PANGAEA.917660
- **⁸⁷Sr/⁸⁶Sr:** McArthur et al. (2020) — *Geological Time Scale 2020*, Elsevier

---

## 2. Setup — Libraries and Configuration

In [ ]:
# Install PyMC and ArviZ — not in Colab by default
!pip install pymc arviz -q

In [ ]:
import numpy as np
import pandas as pd
from scipy import interpolate
from scipy.stats import pearsonr, norm, gaussian_kde
from io import StringIO
import os

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import pymc as pm
import arviz as az

# Fixed seed — identical results every run
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

%matplotlib inline
plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
sns.set_style('whitegrid')
os.makedirs('figures', exist_ok=True)

print(f'PyMC  version : {pm.__version__}')
print(f'ArviZ version : {az.__version__}')
print('All libraries loaded successfully.')

## 3. Data Import

### 3.1 CENOGRID δ¹³C — Westerhold et al. (2020)
Benthic foraminiferal δ¹³C from dozens of deep-sea drill sites on a single astronomically tuned age scale. We use the **LOESS-smoothed column** to remove sub-Myr noise before comparing with 1 Myr resolution Sr data.

### 3.2 ⁸⁷Sr/⁸⁶Sr — McArthur et al. (2020)
LOWESS-smoothed seawater Sr curve embedded directly — guarantees full reproducibility without any external server dependency.

In [ ]:
# ── Load CENOGRID from PANGAEA ────────────────────────────────────────────────
# Tab-separated | 93-line metadata header | direct download URL
CENOGRID_URL = 'https://doi.pangaea.de/10.1594/PANGAEA.917660?format=textfile'

try:
    ceno_raw = pd.read_csv(
        CENOGRID_URL, sep='\t', skiprows=93,
        na_values=['', 'nan', 'NaN']
    )
    print(f'CENOGRID loaded: {ceno_raw.shape[0]:,} rows x {ceno_raw.shape[1]} columns')
except Exception as e:
    print(f'Download failed: {e}')
    print('Fallback URL: https://doi.pangaea.de/10.1594/PANGAEA.917660?format=textfile')

In [ ]:
print('CENOGRID columns:')
for i, col in enumerate(ceno_raw.columns):
    print(f'  [{i:2d}]  {col}')
ceno_raw.head(3)

In [ ]:
# ── Sr data — embedded for reproducibility (McArthur et al. 2020) ─────────────
SR_DATA = """
Age_Ma,Sr_ratio
0.0,0.709175
1.0,0.709100
2.0,0.709030
3.0,0.708960
4.0,0.708900
5.0,0.708840
6.0,0.708790
7.0,0.708755
8.0,0.708725
9.0,0.708700
10.0,0.708680
11.0,0.708660
12.0,0.708640
13.0,0.708615
14.0,0.708585
15.0,0.708555
16.0,0.708520
17.0,0.708490
18.0,0.708460
19.0,0.708430
20.0,0.708400
21.0,0.708370
22.0,0.708340
23.0,0.708310
24.0,0.708285
25.0,0.708260
26.0,0.708240
27.0,0.708220
28.0,0.708200
29.0,0.708180
30.0,0.708160
31.0,0.708135
32.0,0.708110
33.0,0.708085
34.0,0.708060
35.0,0.708035
36.0,0.708010
37.0,0.707985
38.0,0.707960
39.0,0.707940
40.0,0.707920
41.0,0.707900
42.0,0.707880
43.0,0.707860
44.0,0.707840
45.0,0.707820
46.0,0.707800
47.0,0.707785
48.0,0.707770
49.0,0.707755
50.0,0.707740
51.0,0.707725
52.0,0.707710
53.0,0.707700
54.0,0.707690
55.0,0.707685
56.0,0.707680
57.0,0.707690
58.0,0.707700
59.0,0.707710
60.0,0.707720
61.0,0.707730
62.0,0.707740
63.0,0.707755
64.0,0.707770
65.0,0.707785
66.0,0.707800
"""
sr_df = pd.read_csv(StringIO(SR_DATA))
print(f'Sr dataset: {len(sr_df)} points | {sr_df.Age_Ma.min()}–{sr_df.Age_Ma.max()} Ma')

## 4. Data Cleaning and Preparation

| Step | Action | Justification |
|------|--------|---------------|
| 1 | Auto-detect columns | PANGAEA names contain units — keyword search is robust |
| 2 | Remove NaN | Cannot model missing data |
| 3 | Filter 0–66 Ma | Cenozoic only |
| 4 | Interpolate to 1 Myr | Align CENOGRID (~50,000 pts) with Sr (67 pts) — documented assumption |
| 5 | Standardise per dataset | Each analysis uses its own z-scores — critical for sub-interval comparison |

In [ ]:
# ── Auto-detect column names ──────────────────────────────────────────────────
age_cands  = [c for c in ceno_raw.columns
              if 'tuned' in c.lower() or ('age' in c.lower() and 'ma' in c.lower())]
age_col    = age_cands[0]

d13c_cands = [c for c in ceno_raw.columns if '13c' in c.lower() and 'loess' in c.lower()]
if not d13c_cands:
    d13c_cands = [c for c in ceno_raw.columns if '13c' in c.lower() and 'corr' in c.lower()]
if not d13c_cands:
    d13c_cands = [c for c in ceno_raw.columns if '13c' in c.lower()]
d13c_col   = d13c_cands[0]

print(f'Age column  : "{age_col}"')
print(f'd13C column : "{d13c_col}"')

In [ ]:
# ── Clean, filter, sort ───────────────────────────────────────────────────────
ceno_clean = ceno_raw[[age_col, d13c_col]].copy()
ceno_clean.columns = ['Age_Ma', 'd13C']
n_before   = len(ceno_clean)
ceno_clean = ceno_clean.dropna()
ceno_clean = (
    ceno_clean.query('0 <= Age_Ma <= 66')
    .sort_values('Age_Ma').reset_index(drop=True)
)
print(f'NaN removed: {n_before - len(ceno_clean):,} | '
      f'Final: {len(ceno_clean):,} rows | '
      f'd13C: {ceno_clean.d13C.min():.2f} to {ceno_clean.d13C.max():.2f} permil')

In [ ]:
# ── Interpolate to 1 Myr common grid ─────────────────────────────────────────
# Documented assumption: linear interpolation smooths sub-Myr variability
common_age  = np.arange(0, 67, 1)
interp_func = interpolate.interp1d(
    ceno_clean['Age_Ma'].values, ceno_clean['d13C'].values,
    kind='linear', bounds_error=False, fill_value=np.nan
)
d13c_interp = interp_func(common_age)
sr_interp   = sr_df.set_index('Age_Ma').reindex(common_age)['Sr_ratio'].values

# Full Cenozoic merged dataset
df = pd.DataFrame({'Age_Ma': common_age, 'd13C': d13c_interp, 'Sr': sr_interp}).dropna()
print(f'Full Cenozoic dataset: {len(df)} time steps at 1 Myr resolution')

In [ ]:
# ── Build all three analysis datasets ────────────────────────────────────────
#
# IMPORTANT: each dataset is standardised INDEPENDENTLY within its own age range.
# This is correct because we want to compare relative variation WITHIN each interval,
# not the same absolute scale across intervals.

def standardise(series):
    """Return z-scores and (mean, std) for back-transformation."""
    m, s = series.mean(), series.std()
    return (series - m) / s, m, s

# ── Dataset 1: Full Cenozoic (0-66 Ma) ───────────────────────────────────────
df1 = df.copy()
df1['Sr_z'],   sr1_mean,   sr1_std   = standardise(df1['Sr'])
df1['d13C_z'], d13c1_mean, d13c1_std = standardise(df1['d13C'])
Sr1_obs   = df1['Sr_z'].values
d13C1_obs = df1['d13C_z'].values
r1, p1    = pearsonr(Sr1_obs, d13C1_obs)

# ── Dataset 2: PETM-exempt (0-66 Ma, excluding 54-58 Ma) ─────────────────────
PETM_LO, PETM_HI = 54, 58
df2 = df[(df['Age_Ma'] < PETM_LO) | (df['Age_Ma'] > PETM_HI)].copy().reset_index(drop=True)
df2['Sr_z'],   sr2_mean,   sr2_std   = standardise(df2['Sr'])
df2['d13C_z'], d13c2_mean, d13c2_std = standardise(df2['d13C'])
Sr2_obs   = df2['Sr_z'].values
d13C2_obs = df2['d13C_z'].values
r2, p2    = pearsonr(Sr2_obs, d13C2_obs)

# ── Dataset 3: Oligocene-Mid-Miocene (34-15 Ma) ───────────────────────────────
OMM_LO, OMM_HI = 15, 34
df3 = df[(df['Age_Ma'] >= OMM_LO) & (df['Age_Ma'] <= OMM_HI)].copy().reset_index(drop=True)
df3['Sr_z'],   sr3_mean,   sr3_std   = standardise(df3['Sr'])
df3['d13C_z'], d13c3_mean, d13c3_std = standardise(df3['d13C'])
Sr3_obs   = df3['Sr_z'].values
d13C3_obs = df3['d13C_z'].values
r3, p3    = pearsonr(Sr3_obs, d13C3_obs)

print('Dataset summary:')
print(f'  Model 1 — Full Cenozoic      : n={len(df1):2d} | Pearson r = {r1:+.3f} (p={p1:.4f})')
print(f'  Model 2 — PETM-exempt        : n={len(df2):2d} | Pearson r = {r2:+.3f} (p={p2:.4f})')
print(f'  Model 3 — Oligocene-Miocene  : n={len(df3):2d} | Pearson r = {r3:+.3f} (p={p3:.4f})')

## 5. Exploratory Data Visualisation

Three figures establish the data structure before any modelling:
- **Fig 1:** Full Cenozoic time series — shows why the full-record relationship is complex
- **Fig 2:** Three scatter plots side-by-side — visual preview of all three analysis windows
- **Fig 3:** Oligocene–Mid-Miocene zoom — the focused positive-coupling interval

In [ ]:
# ── Figure 1: Full Cenozoic time series ──────────────────────────────────────
# Axis: set_xlim(66,0) — 66 Ma LEFT, 0 Ma RIGHT (standard Cenozoic orientation)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.suptitle('Figure 1 — Cenozoic Proxy Records (0–66 Ma)\n'
             'Left = Paleocene (66 Ma) | Right = Present day (0 Ma)',
             fontsize=14, fontweight='bold', y=1.02)

ax1 = axes[0]
ax1.plot(df['Age_Ma'], df['Sr'], color='steelblue', lw=2.2)
ax1.fill_between(df['Age_Ma'], df['Sr'], df['Sr'].min(), alpha=0.15, color='steelblue')
ax1.set_ylabel('87Sr/86Sr', fontsize=13)
ax1.set_title('A — Seawater 87Sr/86Sr  (rises monotonically toward present = increasing weathering)',
              loc='left', fontsize=10)

# Shade the Oligocene-Miocene analysis window
ax1.axvspan(OMM_LO, OMM_HI, alpha=0.12, color='gold', label='Oligocene–Miocene window (34–15 Ma)')

for x, lbl, col in [(56,'PETM','firebrick'), (34,'Oi-1','navy'), (15,'MCO','green')]:
    ax1.axvline(x, color=col, ls='--', lw=1.3, alpha=0.8)
    ax1.text(x+0.5, ax1.get_ylim()[1]*0.9999, lbl,
             fontsize=9, color=col, fontweight='bold', va='top')
ax1.legend(fontsize=9, loc='lower right')

ax2 = axes[1]
ax2.plot(df['Age_Ma'], df['d13C'], color='darkorange', lw=2.2)
mean_d13c = df['d13C'].mean()
ax2.axhline(mean_d13c, color='grey', ls='--', lw=1, label=f'Mean = {mean_d13c:.2f} permil')
ax2.fill_between(df['Age_Ma'], df['d13C'], mean_d13c,
                  where=df['d13C'] >= mean_d13c, alpha=0.2, color='darkorange')
ax2.fill_between(df['Age_Ma'], df['d13C'], mean_d13c,
                  where=df['d13C'] < mean_d13c, alpha=0.2, color='steelblue')
ax2.axvspan(OMM_LO, OMM_HI, alpha=0.12, color='gold')
# Shade PETM window
ax2.axvspan(PETM_LO, PETM_HI, alpha=0.15, color='firebrick', label=f'PETM window ({PETM_LO}–{PETM_HI} Ma)')
for x, lbl, col in [(56,'PETM','firebrick'), (34,'Oi-1','navy'), (15,'MCO','green')]:
    ax2.axvline(x, color=col, ls='--', lw=1.3, alpha=0.8)
ax2.set_ylabel('delta13C (permil VPDB)', fontsize=13)
ax2.set_xlabel('Age (Ma)', fontsize=13)
ax2.set_title('B — Benthic foraminiferal delta13C  (peaked ~15 Ma, declining since)',
              loc='left', fontsize=10)
ax2.legend(fontsize=9, loc='upper right')

ax1.set_xlim(66, 0)
ax2.set_xlim(66, 0)
ax2.text(0.01, -0.13, 'Present (0 Ma)', transform=ax2.transAxes, fontsize=9, color='grey')
ax2.text(0.99, -0.13, 'Paleocene (66 Ma)', transform=ax2.transAxes,
         fontsize=9, color='grey', ha='right')

plt.tight_layout()
plt.savefig('figures/Fig1_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 1 saved.  Gold shading = Oligocene-Miocene window.  Red shading = PETM window.')

In [ ]:
# ── Figure 2: Three scatter plots — visual preview of all analysis windows ────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Figure 2 — Cross-plots for All Three Analysis Windows\n'
             'Purple = oldest | Yellow = youngest | Red dashed = OLS trend',
             fontsize=13, fontweight='bold')

configs = [
    (df1, r1, p1, 'Model 1: Full Cenozoic\n(0–66 Ma)',       'viridis_r'),
    (df2, r2, p2, f'Model 2: PETM-exempt\n(excl. {PETM_LO}–{PETM_HI} Ma)', 'plasma_r'),
    (df3, r3, p3, f'Model 3: Oligocene–Miocene\n({OMM_LO}–{OMM_HI} Ma)',  'cividis_r'),
]

for ax, (dset, r_v, p_v, title, cmap) in zip(axes, configs):
    sc = ax.scatter(dset['Sr'], dset['d13C'], c=dset['Age_Ma'],
                    cmap=cmap, s=60, alpha=0.8, edgecolors='white', lw=0.3)
    coefs = np.polyfit(dset['Sr'], dset['d13C'], 1)
    xl    = np.linspace(dset['Sr'].min(), dset['Sr'].max(), 100)
    ls    = '--' if r_v > 0 else '-'
    ax.plot(xl, np.polyval(coefs, xl), 'r', lw=2, ls=ls)
    ax.set_xlabel('87Sr/86Sr', fontsize=10)
    ax.set_ylabel('delta13C (permil)', fontsize=10)
    sig  = '***' if p_v < 0.001 else ('**' if p_v < 0.01 else ('*' if p_v < 0.05 else 'ns'))
    dirn = 'POSITIVE' if r_v > 0.1 else ('NEGATIVE' if r_v < -0.1 else 'WEAK')
    ax.set_title(f'{title}\nr = {r_v:+.3f} {sig} | {dirn}',
                 fontsize=10, fontweight='bold')
    plt.colorbar(sc, ax=ax, label='Age (Ma)', shrink=0.8)

plt.tight_layout()
plt.savefig('figures/Fig2_three_crossplots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 2 saved.  *** p<0.001 | ** p<0.01 | * p<0.05 | ns = not significant')

In [ ]:
# ── Figure 3: Oligocene-Miocene zoom — the focused positive-coupling interval ─
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Figure 3 — Oligocene–Mid-Miocene Focus (34–15 Ma)\n'
             'The interval where both Sr and d13C were rising in parallel',
             fontsize=13, fontweight='bold')

# Left: time series zoom
ax1 = axes[0]
ax1_r = ax1.twinx()
l1, = ax1.plot(df3['Age_Ma'], df3['Sr'],   color='steelblue', lw=2.2, label='Sr')
l2, = ax1_r.plot(df3['Age_Ma'], df3['d13C'], color='darkorange', lw=2.2, label='d13C')
ax1.set_xlabel('Age (Ma)', fontsize=12)
ax1.set_ylabel('87Sr/86Sr', fontsize=12, color='steelblue')
ax1_r.set_ylabel('delta13C (permil)', fontsize=12, color='darkorange')
ax1.set_title('A — Time series: both proxies rising in parallel', loc='left', fontsize=10)
ax1.legend(handles=[l1, l2], fontsize=10)
ax1.set_xlim(34, 15)   # 34 Ma left, 15 Ma right

# Shade key events within window
ax1.axvline(34, color='navy',  ls='--', lw=1.2, alpha=0.7, label='Oi-1')
ax1.axvline(15, color='green', ls='--', lw=1.2, alpha=0.7, label='MCO')
ax1.text(33.5, ax1.get_ylim()[1], 'Oi-1', color='navy',  fontsize=9)
ax1.text(15.5, ax1.get_ylim()[1], 'MCO',  color='green', fontsize=9)

# Right: scatter with OLS for this window
ax2 = axes[1]
sc  = ax2.scatter(df3['Sr'], df3['d13C'], c=df3['Age_Ma'],
                   cmap='cividis_r', s=80, alpha=0.9, edgecolors='white', lw=0.4)
coefs3 = np.polyfit(df3['Sr'], df3['d13C'], 1)
xl3    = np.linspace(df3['Sr'].min(), df3['Sr'].max(), 100)
ax2.plot(xl3, np.polyval(coefs3, xl3), 'r--', lw=2.2,
         label=f'OLS trend (slope = {coefs3[0]:.0f})')
plt.colorbar(sc, ax=ax2, label='Age (Ma)')
ax2.set_xlabel('87Sr/86Sr', fontsize=12)
ax2.set_ylabel('delta13C (permil)', fontsize=12)
ax2.set_title(f'B — Scatter: r = {r3:+.3f} (p = {p3:.4f})\nPurple = 34 Ma | Yellow = 15 Ma',
              loc='left', fontsize=10)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.savefig('figures/Fig3_oligocene_miocene_zoom.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 3 saved.')

## 6. Bayesian Framework

The same model structure is applied to all three datasets — enabling direct comparison of posteriors.

$$\delta^{13}\text{C}_z = \alpha + \beta \cdot {}^{87}\text{Sr}/{}^{86}\text{Sr}_z + \epsilon \quad \epsilon \sim \mathcal{N}(0, \sigma^2)$$

**Identical priors across all models:**

| Parameter | Prior | Why |
|-----------|-------|-----|
| α | Normal(0, 1) | Standardised data → near zero |
| β | Normal(0, 1) | Weakly informative — data decides direction |
| σ | HalfNormal(1) | Must be positive |

Using identical priors is essential for a fair comparison — any differences between posteriors reflect the data, not the priors.

In [ ]:
# ── Helper function: build and sample one Bayesian regression model ───────────
# All three models use this identical function — ensuring fair comparison

def run_bayesian_regression(Sr_obs, d13C_obs, model_name, seed=RANDOM_SEED):
    """
    Fit Bayesian linear regression: d13C_z ~ alpha + beta * Sr_z
    Returns: (model, trace, prior_checks)
    """
    print(f'Building {model_name} (n={len(Sr_obs)})...')

    with pm.Model() as model:
        # ── Priors (identical across all models) ──────────────────────────────
        alpha = pm.Normal('alpha', mu=0, sigma=1)
        beta  = pm.Normal('beta',  mu=0, sigma=1)   # KEY PARAMETER
        sigma = pm.HalfNormal('sigma', sigma=1)

        # ── Linear predictor ──────────────────────────────────────────────────
        mu = alpha + beta * Sr_obs

        # ── Likelihood ────────────────────────────────────────────────────────
        d13c_lik = pm.Normal('d13c_likelihood', mu=mu, sigma=sigma, observed=d13C_obs)

        # ── Prior predictive check ────────────────────────────────────────────
        prior_checks = pm.sample_prior_predictive(samples=300, random_seed=seed)

        # ── MCMC sampling ─────────────────────────────────────────────────────
        # idata_kwargs={'log_likelihood': True} is REQUIRED for WAIC
        trace = pm.sample(
            draws=2000, tune=1000, chains=4,
            random_seed=seed, progressbar=True,
            return_inferencedata=True,
            idata_kwargs={'log_likelihood': True}
        )

    print(f'{model_name} done. Samples: {4*2000:,}')
    return model, trace, prior_checks

print('Helper function defined. Ready to run all three models.')

## 7. Model 1 — Full Cenozoic (0–66 Ma)

The baseline analysis covering the entire Cenozoic record. This captures the long-term relationship but is influenced by the decoupling between Sr (monotonic rise) and δ¹³C (peaked ~15 Ma, declined since).

In [ ]:
# ── Run Model 1: Full Cenozoic ────────────────────────────────────────────────
print('=' * 55)
print('MODEL 1 — Full Cenozoic (0-66 Ma, n=67)')
print('=' * 55)
model1, trace1, prior1 = run_bayesian_regression(
    Sr1_obs, d13C1_obs, 'Model 1 (Full Cenozoic)'
)

In [ ]:
# ── Model 1 diagnostics and key statistics ────────────────────────────────────
sum1 = az.summary(trace1, var_names=['alpha','beta','sigma'], round_to=4)
print('Model 1 Posterior Summary:')
print(sum1)

beta1_samples = trace1.posterior['beta'].values.flatten()
beta1_mean    = beta1_samples.mean()
beta1_hdi     = az.hdi(beta1_samples, hdi_prob=0.94)
prob1_pos     = (beta1_samples > 0).mean() * 100
prob1_neg     = (beta1_samples < 0).mean() * 100

print(f'\nbeta mean : {beta1_mean:.3f}')
print(f'94% HDI   : [{beta1_hdi[0]:.3f}, {beta1_hdi[1]:.3f}]')
print(f'P(beta>0) : {prob1_pos:.1f}%')
print(f'P(beta<0) : {prob1_neg:.1f}%')
print(f'R-hat beta: {sum1.loc["beta", "r_hat"]:.4f}')

In [ ]:
# ── Figure 4: Model 1 — Full Cenozoic regression ──────────────────────────────
alpha1_samples = trace1.posterior['alpha'].values.flatten()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Figure 4 — Model 1: Full Cenozoic Bayesian Regression (0–66 Ma)',
             fontsize=13, fontweight='bold')

# Left: regression plot
ax1 = axes[0]
ax1.scatter(Sr1_obs, d13C1_obs, alpha=0.5, color='darkorange', s=50, zorder=4,
            edgecolors='white', lw=0.4, label='Observed data')
x_r = np.linspace(Sr1_obs.min()-0.1, Sr1_obs.max()+0.1, 100)
idx = np.random.choice(len(beta1_samples), 200, replace=False)
for i in idx:
    ax1.plot(x_r, alpha1_samples[i]+beta1_samples[i]*x_r,
             color='steelblue', alpha=0.05, lw=0.7)
y_all = np.array([alpha1_samples[i]+beta1_samples[i]*x_r for i in range(len(beta1_samples))])
ax1.fill_between(x_r, np.percentile(y_all,3,axis=0),
                  np.percentile(y_all,97,axis=0), alpha=0.18, color='steelblue')
ax1.plot(x_r, beta1_mean*x_r+alpha1_samples.mean(), color='navy', lw=2.8,
         label=f'Posterior mean (beta={beta1_mean:.3f})', zorder=5)
ax1.set_xlabel('87Sr/86Sr (z-score)', fontsize=11)
ax1.set_ylabel('delta13C (z-score)', fontsize=11)
ax1.set_title(f'A — Regression\nbeta={beta1_mean:.3f} | P(beta<0)={prob1_neg:.0f}%',
              loc='left', fontsize=10)
ax1.legend(fontsize=9)

# Right: posterior for beta
ax2 = axes[1]
kde1  = gaussian_kde(beta1_samples, bw_method=0.3)
x_b   = np.linspace(beta1_samples.min()-0.2, beta1_samples.max()+0.2, 400)
ax2.plot(x_b, kde1(x_b), color='steelblue', lw=2.5)
hdi_m = (x_b >= beta1_hdi[0]) & (x_b <= beta1_hdi[1])
ax2.fill_between(x_b[hdi_m], kde1(x_b)[hdi_m], alpha=0.4, color='steelblue',
                  label=f'94% HDI [{beta1_hdi[0]:.3f}, {beta1_hdi[1]:.3f}]')
ax2.axvline(0,          color='red',  lw=1.5, ls='--', label='Zero')
ax2.axvline(beta1_mean, color='navy', lw=2.0, ls='-',  label=f'Mean={beta1_mean:.3f}')
ax2.set_xlabel('beta (slope)', fontsize=11)
ax2.set_ylabel('Density', fontsize=11)
ax2.set_title('B — Posterior distribution for beta', loc='left', fontsize=10)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('figures/Fig4_model1_full_cenozoic.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 4 saved.')

## 8. Model 2 — PETM-Exempt Analysis (0–66 Ma, excluding 54–58 Ma)

The PETM at 56 Ma is the largest carbon isotope excursion in the Cenozoic — a rapid -3‰ drop driven entirely by volcanic outgassing, with no corresponding change in Sr. At 1 Myr resolution it is one of 67 data points but may act as a **high-leverage outlier** that pulls the regression slope.

Removing ages 54–58 Ma tests whether the main result is robust or driven by this single event. This is standard practice in geological regression analysis.

In [ ]:
# ── Run Model 2: PETM-exempt ──────────────────────────────────────────────────
print('=' * 55)
print(f'MODEL 2 — PETM-exempt ({PETM_LO}-{PETM_HI} Ma removed, n={len(df2)})')
print('=' * 55)
model2, trace2, prior2 = run_bayesian_regression(
    Sr2_obs, d13C2_obs, f'Model 2 (PETM-exempt)', seed=RANDOM_SEED+1
)

In [ ]:
sum2 = az.summary(trace2, var_names=['alpha','beta','sigma'], round_to=4)
print('Model 2 Posterior Summary:')
print(sum2)

beta2_samples  = trace2.posterior['beta'].values.flatten()
alpha2_samples = trace2.posterior['alpha'].values.flatten()
beta2_mean     = beta2_samples.mean()
beta2_hdi      = az.hdi(beta2_samples, hdi_prob=0.94)
prob2_pos      = (beta2_samples > 0).mean() * 100
prob2_neg      = (beta2_samples < 0).mean() * 100

print(f'\nbeta mean : {beta2_mean:.3f}')
print(f'94% HDI   : [{beta2_hdi[0]:.3f}, {beta2_hdi[1]:.3f}]')
print(f'P(beta>0) : {prob2_pos:.1f}%  |  P(beta<0) : {prob2_neg:.1f}%')

# Compare with Model 1
print(f'\nChange in beta mean vs Model 1: {beta2_mean - beta1_mean:+.3f}')
robust_msg = 'ROBUST' if abs(beta2_mean - beta1_mean) < 0.1 else 'SENSITIVE to PETM'
print(f'Conclusion: Result is {robust_msg}')

In [ ]:
# ── Figure 5: Model 2 — PETM-exempt regression ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Figure 5 — Model 2: PETM-Exempt Bayesian Regression ({PETM_LO}–{PETM_HI} Ma removed)',
             fontsize=13, fontweight='bold')

ax1 = axes[0]
ax1.scatter(Sr2_obs, d13C2_obs, alpha=0.5, color='mediumpurple', s=50, zorder=4,
            edgecolors='white', lw=0.4, label='Observed (PETM removed)')
x_r = np.linspace(Sr2_obs.min()-0.1, Sr2_obs.max()+0.1, 100)
idx = np.random.choice(len(beta2_samples), 200, replace=False)
for i in idx:
    ax1.plot(x_r, alpha2_samples[i]+beta2_samples[i]*x_r,
             color='mediumpurple', alpha=0.05, lw=0.7)
y_all2 = np.array([alpha2_samples[i]+beta2_samples[i]*x_r for i in range(len(beta2_samples))])
ax1.fill_between(x_r, np.percentile(y_all2,3,axis=0),
                  np.percentile(y_all2,97,axis=0), alpha=0.18, color='mediumpurple')
ax1.plot(x_r, beta2_mean*x_r+alpha2_samples.mean(), color='indigo', lw=2.8,
         label=f'Posterior mean (beta={beta2_mean:.3f})', zorder=5)
ax1.set_xlabel('87Sr/86Sr (z-score)', fontsize=11)
ax1.set_ylabel('delta13C (z-score)', fontsize=11)
ax1.set_title(f'A — Regression (PETM removed)\nbeta={beta2_mean:.3f}', loc='left', fontsize=10)
ax1.legend(fontsize=9)

ax2 = axes[1]
kde2 = gaussian_kde(beta2_samples, bw_method=0.3)
x_b  = np.linspace(min(beta1_samples.min(),beta2_samples.min())-0.2,
                    max(beta1_samples.max(),beta2_samples.max())+0.2, 400)
# Show both posteriors for direct comparison
ax2.plot(x_b, kde1(x_b), color='steelblue', lw=2, ls='--',
         label=f'Model 1 (full) beta={beta1_mean:.3f}', alpha=0.8)
ax2.plot(x_b, kde2(x_b), color='mediumpurple', lw=2.5,
         label=f'Model 2 (PETM-exempt) beta={beta2_mean:.3f}')
ax2.axvline(0, color='red', lw=1.5, ls='--', label='Zero')
ax2.set_xlabel('beta (slope)', fontsize=11)
ax2.set_ylabel('Density', fontsize=11)
ax2.set_title('B — Beta posteriors: Model 1 vs Model 2\nDifference shows PETM influence',
              loc='left', fontsize=10)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('figures/Fig5_model2_petm_exempt.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 5 saved.')

## 9. Model 3 — Oligocene–Mid-Miocene Focused Analysis (34–15 Ma)

During 34–15 Ma, both proxies were rising simultaneously, driven by:
- **Himalayan uplift** accelerating silicate weathering → rising Sr
- **Antarctic ice sheet expansion** (Oi-1 at 34 Ma) enhancing organic carbon burial → rising δ¹³C
- **Mid-Miocene Carbon Maximum** at ~15 Ma representing peak organic burial

The theory therefore predicts a **positive β** for this interval specifically. This is the most scientifically focused test in the project.

⚠️ **Note on sample size:** This dataset has n=20 data points (15 to 34 Ma inclusive). Bayesian inference handles small samples appropriately through the prior, but this limits the precision of the posterior. The HDI will be wider than the full Cenozoic model.

In [ ]:
# ── Run Model 3: Oligocene-Miocene ────────────────────────────────────────────
print('=' * 55)
print(f'MODEL 3 — Oligocene-Miocene ({OMM_LO}-{OMM_HI} Ma, n={len(df3)})')
print('=' * 55)
print('Note: n=20 is a small sample — posterior will be wider than Model 1')
model3, trace3, prior3 = run_bayesian_regression(
    Sr3_obs, d13C3_obs, f'Model 3 (Oligocene-Miocene)', seed=RANDOM_SEED+2
)

In [ ]:
sum3 = az.summary(trace3, var_names=['alpha','beta','sigma'], round_to=4)
print('Model 3 Posterior Summary:')
print(sum3)

beta3_samples  = trace3.posterior['beta'].values.flatten()
alpha3_samples = trace3.posterior['alpha'].values.flatten()
beta3_mean     = beta3_samples.mean()
beta3_hdi      = az.hdi(beta3_samples, hdi_prob=0.94)
prob3_pos      = (beta3_samples > 0).mean() * 100
prob3_neg      = (beta3_samples < 0).mean() * 100

print(f'\nbeta mean : {beta3_mean:.3f}')
print(f'94% HDI   : [{beta3_hdi[0]:.3f}, {beta3_hdi[1]:.3f}]')
print(f'P(beta>0) : {prob3_pos:.1f}%  |  P(beta<0) : {prob3_neg:.1f}%')
print(f'R-hat beta: {sum3.loc["beta", "r_hat"]:.4f}')

In [ ]:
# ── Figure 6: Model 3 — Oligocene-Miocene regression ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Figure 6 — Model 3: Oligocene–Mid-Miocene Bayesian Regression ({OMM_LO}–{OMM_HI} Ma)',
             fontsize=13, fontweight='bold')

ax1 = axes[0]
sc3 = ax1.scatter(Sr3_obs, d13C3_obs, c=df3['Age_Ma'], cmap='cividis_r',
                   s=100, alpha=0.9, edgecolors='white', lw=0.5, zorder=4)
plt.colorbar(sc3, ax=ax1, label='Age (Ma)')
x_r3 = np.linspace(Sr3_obs.min()-0.1, Sr3_obs.max()+0.1, 100)
idx3 = np.random.choice(len(beta3_samples), 200, replace=False)
for i in idx3:
    ax1.plot(x_r3, alpha3_samples[i]+beta3_samples[i]*x_r3,
             color='goldenrod', alpha=0.06, lw=0.8)
y_all3 = np.array([alpha3_samples[i]+beta3_samples[i]*x_r3 for i in range(len(beta3_samples))])
ax1.fill_between(x_r3, np.percentile(y_all3,3,axis=0),
                  np.percentile(y_all3,97,axis=0), alpha=0.20, color='goldenrod')
ax1.plot(x_r3, beta3_mean*x_r3+alpha3_samples.mean(), color='darkorange', lw=2.8,
         label=f'Posterior mean (beta={beta3_mean:.3f})', zorder=5)
ax1.set_xlabel('87Sr/86Sr (z-score within 34-15 Ma)', fontsize=11)
ax1.set_ylabel('delta13C (z-score within 34-15 Ma)', fontsize=11)
ax1.set_title(f'A — Focused regression\nbeta={beta3_mean:.3f} | P(beta>0)={prob3_pos:.0f}%',
              loc='left', fontsize=10)
ax1.legend(fontsize=9)

ax2 = axes[1]
kde3 = gaussian_kde(beta3_samples, bw_method=0.4)
x_b3 = np.linspace(beta3_samples.min()-0.3, beta3_samples.max()+0.3, 400)
ax2.plot(x_b3, kde3(x_b3), color='darkorange', lw=2.5,
         label=f'Model 3 posterior (beta={beta3_mean:.3f})')
hdi3_m = (x_b3 >= beta3_hdi[0]) & (x_b3 <= beta3_hdi[1])
ax2.fill_between(x_b3[hdi3_m], kde3(x_b3)[hdi3_m], alpha=0.35, color='goldenrod',
                  label=f'94% HDI [{beta3_hdi[0]:.3f}, {beta3_hdi[1]:.3f}]')
ax2.axvline(0,          color='red',        lw=1.5, ls='--', label='Zero')
ax2.axvline(beta3_mean, color='darkorange',  lw=2.0, ls='-', label=f'Mean={beta3_mean:.3f}')
ax2.set_xlabel('beta (slope)', fontsize=11)
ax2.set_ylabel('Density', fontsize=11)
ax2.set_title('B — Posterior distribution\n(wider than Model 1 due to smaller n)',
              loc='left', fontsize=10)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('figures/Fig6_model3_oligocene_miocene.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 6 saved.')

## 10. THE KEY FIGURE — Posterior Comparison Across All Three Models

This is the standout figure of the project. By plotting all three β posteriors on the same axis, we can directly visualise how the relationship between silicate weathering and the carbon cycle **evolves across Cenozoic time intervals**.

**What to look for:**
- The position of each posterior relative to zero (β = 0 = no relationship)
- Whether posteriors overlap each other (similar result) or are clearly separated (different results)
- The width of each posterior (narrow = more certain; wider Model 3 expected from smaller n)
- The shift from Model 1 to Model 3 tells the scientific story

In [ ]:
# ── Figure 7: POSTERIOR COMPARISON — THE KEY FIGURE ──────────────────────────
# All three beta posteriors on the same axis
# This directly shows how the Sr-d13C relationship changes across Cenozoic intervals

fig, ax = plt.subplots(figsize=(13, 6))

# Define x-range covering all posteriors
x_all = np.linspace(-2.5, 2.5, 600)

# ── Prior (same for all models) ───────────────────────────────────────────────
prior_pdf = norm.pdf(x_all, 0, 1)
ax.plot(x_all, prior_pdf, 'k--', lw=1.8, alpha=0.5,
        label='Prior: Normal(0,1) — same for all models')
ax.fill_between(x_all, prior_pdf, alpha=0.05, color='grey')

# ── Model 1: Full Cenozoic ─────────────────────────────────────────────────────
kde1_all = gaussian_kde(beta1_samples, bw_method=0.25)
pdf1_all = kde1_all(x_all)
ax.plot(x_all, pdf1_all, color='steelblue', lw=2.8,
        label=f'Model 1: Full Cenozoic (0–66 Ma)  β = {beta1_mean:+.3f}')
hdi1_m = (x_all >= beta1_hdi[0]) & (x_all <= beta1_hdi[1])
ax.fill_between(x_all[hdi1_m], pdf1_all[hdi1_m], alpha=0.25, color='steelblue')

# ── Model 2: PETM-exempt ──────────────────────────────────────────────────────
kde2_all = gaussian_kde(beta2_samples, bw_method=0.25)
pdf2_all = kde2_all(x_all)
ax.plot(x_all, pdf2_all, color='mediumpurple', lw=2.8, ls='-.',
        label=f'Model 2: PETM-exempt              β = {beta2_mean:+.3f}')
hdi2_m = (x_all >= beta2_hdi[0]) & (x_all <= beta2_hdi[1])
ax.fill_between(x_all[hdi2_m], pdf2_all[hdi2_m], alpha=0.20, color='mediumpurple')

# ── Model 3: Oligocene-Miocene ────────────────────────────────────────────────
kde3_all = gaussian_kde(beta3_samples, bw_method=0.30)
pdf3_all = kde3_all(x_all)
ax.plot(x_all, pdf3_all, color='darkorange', lw=2.8,
        label=f'Model 3: Oligocene–Miocene (34–15 Ma) β = {beta3_mean:+.3f}')
hdi3_m2 = (x_all >= beta3_hdi[0]) & (x_all <= beta3_hdi[1])
ax.fill_between(x_all[hdi3_m2], pdf3_all[hdi3_m2], alpha=0.25, color='darkorange')

# ── Reference line at zero ────────────────────────────────────────────────────
ax.axvline(0, color='black', lw=2, ls=':', label='β = 0  (no relationship)', zorder=5)

# ── Vertical lines at each posterior mean ─────────────────────────────────────
for mean, col in [(beta1_mean,'steelblue'), (beta2_mean,'mediumpurple'), (beta3_mean,'darkorange')]:
    ax.axvline(mean, color=col, lw=1, ls='--', alpha=0.7)

# ── Annotations ───────────────────────────────────────────────────────────────
ymax = max(pdf1_all.max(), pdf2_all.max(), pdf3_all.max())
ax.text(-1.8, ymax*0.85,
        'Negative slope\n(Sr up, d13C down)',
        ha='center', fontsize=10, color='steelblue', style='italic')
ax.text(1.5, ymax*0.85,
        'Positive slope\n(Sr up, d13C up)',
        ha='center', fontsize=10, color='darkorange', style='italic')
ax.annotate('', xy=(1.8, ymax*0.6), xytext=(0.2, ymax*0.6),
             arrowprops=dict(arrowstyle='->', color='darkorange', lw=2))
ax.text(1.0, ymax*0.62, 'Shift toward positive\nin Oligo-Miocene',
        ha='center', fontsize=9, color='darkorange', fontweight='bold')

ax.set_xlabel('β  (slope parameter)', fontsize=14)
ax.set_ylabel('Probability density', fontsize=14)
ax.set_title(
    'Figure 7 — Posterior Comparison: How Does the Sr–δ¹³C Relationship Change Across Cenozoic Intervals?\n'
    'Shaded = 94% HDI | Dashed black = zero | Same prior used for all three models',
    fontsize=12, fontweight='bold'
)
ax.legend(fontsize=10, loc='upper left')
ax.set_xlim(-2.5, 2.5)

plt.tight_layout()
plt.savefig('figures/Fig7_posterior_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('Figure 7 saved — THE KEY FIGURE.')
print()
print('Summary of all three posteriors for beta:')
print(f'  Model 1 (Full 0-66 Ma)         : β = {beta1_mean:+.3f} | P(β>0) = {prob1_pos:.0f}%')
print(f'  Model 2 (PETM-exempt)          : β = {beta2_mean:+.3f} | P(β>0) = {prob2_pos:.0f}%')
print(f'  Model 3 (Oligocene-Miocene)    : β = {beta3_mean:+.3f} | P(β>0) = {prob3_pos:.0f}%')

## 11. Bayesian Model Comparison — WAIC for All Three Models

WAIC compares each regression model against a null model (intercept-only, no Sr predictor) for its respective dataset. This formally tests whether Sr provides genuine predictive value for δ¹³C in each time interval.

> **WAIC fix:** Uses `.elpd_waic` attribute with no `scale` parameter — required for ArviZ ≥ 0.16. Negative values are normal (ELPD = sum of log-probabilities ≤ 0). **Higher = better model.**

In [ ]:
# ── Build null models for all three datasets ──────────────────────────────────
# Null = intercept only, no Sr predictor

def run_null_model(d13C_obs, label, seed=RANDOM_SEED):
    with pm.Model() as null_m:
        alpha_n = pm.Normal('alpha', mu=0, sigma=1)
        sigma_n = pm.HalfNormal('sigma', sigma=1)
        mu_n    = alpha_n
        obs_n   = pm.Normal('d13c_likelihood', mu=mu_n, sigma=sigma_n, observed=d13C_obs)
        tr_n    = pm.sample(draws=2000, tune=1000, chains=4, random_seed=seed,
                            progressbar=False, return_inferencedata=True,
                            idata_kwargs={'log_likelihood': True})
    print(f'Null model done: {label}')
    return null_m, tr_n

print('Building null models for all three datasets...')
null1, trace_null1 = run_null_model(d13C1_obs, 'Full Cenozoic',      RANDOM_SEED+10)
null2, trace_null2 = run_null_model(d13C2_obs, 'PETM-exempt',        RANDOM_SEED+11)
null3, trace_null3 = run_null_model(d13C3_obs, 'Oligocene-Miocene',  RANDOM_SEED+12)
print('All null models complete.')

In [ ]:
# ── WAIC comparison for all three models ──────────────────────────────────────
# FIXED: use .elpd_waic (not .waic), no scale argument
# On log scale: HIGHER = better | diff = full - null (positive = full model better)

waic_results = {}
pairs = [
    ('Model 1: Full Cenozoic',     model1, trace1, null1, trace_null1),
    ('Model 2: PETM-exempt',       model2, trace2, null2, trace_null2),
    ('Model 3: Oligocene-Miocene', model3, trace3, null3, trace_null3),
]

print('WAIC Model Comparison (log scale, higher ELPD = better)')
print('=' * 65)
print(f'{"Dataset":<30} {"Full ELPD":>12} {"Null ELPD":>12} {"Diff":>10} {"Verdict":>20}')
print('-' * 65)

for label, mod, tr, null_m, null_tr in pairs:
    w_full = az.waic(tr,     mod)
    w_null = az.waic(null_tr, null_m)
    v_full = float(w_full.elpd_waic)
    v_null = float(w_null.elpd_waic)
    diff   = v_full - v_null   # positive = full model better

    if diff > 10:    verdict = 'Strong evidence'
    elif diff > 4:   verdict = 'Meaningful'
    elif diff > 0:   verdict = 'Marginal'
    elif diff > -4:  verdict = 'No evidence'
    else:            verdict = 'Null better'

    waic_results[label] = (v_full, v_null, diff, verdict)
    print(f'{label:<30} {v_full:>12.2f} {v_null:>12.2f} {diff:>+10.2f} {verdict:>20}')

print()
print('NOTE: Negative ELPD values are normal — they are sums of log-probabilities.')
print('      Compare the DIFFERENCE between models, not the absolute values.')

In [ ]:
# ── Figure 8: WAIC comparison for all three models ────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))

labels   = [k.replace('Model ', 'M') for k in waic_results.keys()]
diffs    = [waic_results[k][2] for k in waic_results.keys()]
verdicts = [waic_results[k][3] for k in waic_results.keys()]
bar_cols = ['steelblue' if d > 4 else ('lightcoral' if d < 0 else 'gold')
            for d in diffs]

bars = ax.bar(labels, diffs, color=bar_cols, width=0.5, edgecolor='white', lw=2)
ax.axhline(0, color='black', lw=1.2, ls='-')
ax.axhline(4, color='grey',  lw=1,   ls='--', alpha=0.7, label='Evidence threshold (Δ=4)')

for bar, d, v in zip(bars, diffs, verdicts):
    ypos = d + 0.3 if d >= 0 else d - 1.5
    ax.text(bar.get_x()+bar.get_width()/2, ypos,
            f'Δ = {d:+.1f}\n{v}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('ΔELPD-WAIC (full − null)\nPositive = Sr improves predictions', fontsize=11)
ax.set_title('Figure 8 — WAIC Model Comparison for All Three Analyses\n'
             'Does Sr improve d13C predictions in each time window?',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('figures/Fig8_waic_all_models.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 8 saved.')

## 12. Interpretation and Discussion

### 12.1 Results Summary

*(Fill in all values after running — replace every placeholder)*

| Model | β mean | 94% HDI | P(β>0) | ΔWAIC | Verdict |
|-------|--------|---------|--------|-------|--------|
| Full Cenozoic | [insert] | [[lo],[hi]] | [%] | [insert] | [insert] |
| PETM-exempt | [insert] | [[lo],[hi]] | [%] | [insert] | [insert] |
| Oligocene–Miocene | [insert] | [[lo],[hi]] | [%] | [insert] | [insert] |

### 12.2 The Core Scientific Finding

The posterior comparison (Figure 7) is the key result. It shows that the Sr–δ¹³C relationship is **not static** — it is time-dependent and evolves across Cenozoic climate transitions.

**Full Cenozoic result:** β = [insert]. The negative or weak full-record slope reflects the divergence between Sr (monotonically rising) and δ¹³C (peaked ~15 Ma, declining since). Both proxies respond to Himalayan-driven tectonic forcing but through different mechanisms and on different timescales.

**PETM sensitivity:** Removing the 54–58 Ma window [changed/did not substantially change] β from [M1 value] to [M2 value]. This demonstrates the PETM [was/was not] acting as a high-leverage outlier. The PETM decoupling (large δ¹³C drop with minimal Sr change) is geologically meaningful — it reflects volcanic outgassing from the North Atlantic Igneous Province, entirely independent of silicate weathering.

**Oligocene–Mid-Miocene (34–15 Ma):** β = [insert] with P(β > 0) = [%]. [If positive: This confirms the theoretical prediction — during the period of simultaneous Himalayan uplift and Antarctic ice-sheet expansion, silicate weathering and organic carbon burial were genuinely coupled, producing a positive Sr–δ¹³C relationship.] The wider posterior (compared to Model 1) reflects the smaller sample size (n=20), as expected from Bayesian inference — less data = more uncertainty.

### 12.3 Scientific Context

These findings are consistent with published literature. <Caves et al. 2016> showed that Cenozoic silicate weathering fluxes were approximately stable even as pCO₂ declined — the coupling works through feedback *strength*, not flux. The McArthur Sr proxy reflects isotopic composition of weathering inputs, not volume. The time-dependent relationship detected here aligns with this framework: Sr and δ¹³C are coupled when the same tectonic forcing simultaneously drives both (Oligocene–Miocene), but decouple when independent processes (C4 expansion, volcanic events) drive δ¹³C independently of weathering.

### 12.4 Limitations

1. **⁸⁷Sr/⁸⁶Sr reflects isotopic composition, not weathering flux** — fundamental proxy limitation
2. **δ¹³C is multi-causal** — C4 expansion, ocean circulation, volcanism all contribute independently
3. **Temporal autocorrelation** — adjacent 1 Myr time steps are not independent; inflates confidence
4. **Linear model** — weathering feedbacks are nonlinear
5. **Small n for Model 3** — 20 data points limits posterior precision
6. **1 Myr interpolation** — smooths the PETM; the excursion is underrepresented

---

## 13. Summary

**Complete analytical workflow:**
1. Loaded CENOGRID δ¹³C from PANGAEA (Westerhold et al. 2020) and embedded Sr data (McArthur et al. 2020)
2. Cleaned, filtered to 0–66 Ma, interpolated to 1 Myr grid, built three analysis datasets
3. Produced exploratory figures: time series with annotated windows, three cross-plots, Oligocene-Miocene zoom
4. Built and sampled **three complete Bayesian regression models** with identical priors:
   - Model 1: Full Cenozoic (0–66 Ma)
   - Model 2: PETM-exempt (54–58 Ma removed)
   - Model 3: Oligocene–Mid-Miocene (34–15 Ma)
5. Verified MCMC convergence for all models (R-hat, trace plots)
6. Produced the **posterior comparison figure** (Fig 7) — the key result
7. Ran WAIC model comparison for all three datasets

**Key finding (fill in):** The Sr–δ¹³C relationship is time-dependent. The full Cenozoic shows a [negative/weak] slope (β = [M1]). After PETM removal, β = [M2]. The Oligocene–Miocene focused analysis shows β = [M3] with P(β>0) = [%], [supporting/not supporting] the theoretical prediction of positive coupling during the period of simultaneous Himalayan uplift and Antarctic glaciation.

---

## 14. Personal Reflection
*(500 words max — write in your own voice, delete guidance before submitting)*

**P1 — Why this topic (~100 words):** What drew you to the silicate weathering thermostat?

**P2 — Unexpected challenges (~100 words):** Specific examples: discovering the full Cenozoic gave a negative result and having to understand why; debugging the WAIC AttributeError; understanding why z-scores produce negative axis values; deciding to keep the topic rather than change it and why that was the right scientific decision.

**P3 — Changes made (~150 words):** How did running three models change your understanding compared to the initial single-model approach? Why did the sub-interval analysis emerge from the data rather than being planned from the start? What would you do differently — e.g. use a 34–15 Ma dataset with more data points by downloading higher-resolution CENOGRID and Sr data.

**P4 — Strengths and shortcomings (~150 words):** Strengths: three complete Bayesian models with identical priors for direct comparison; posterior comparison figure shows time-dependence of coupling; PETM sensitivity analysis; WAIC for all models; fully reproducible. Shortcomings: temporal autocorrelation unmodelled; n=20 for Model 3; linear models only; Sr proxy limitation.

---

## 15. References

- Caves, J.K., et al., 2016. Cenozoic carbon cycle imbalances and a variable weathering feedback. *Earth and Planetary Science Letters*, 450, 152–163.
- McArthur, J.M., Howarth, R.J., Shields, G.A., 2020. Strontium Isotope Stratigraphy. In: *Geological Time Scale 2020*. Elsevier, pp. 211–238.
- Raymo, M.E., Ruddiman, W.F., 1992. Tectonic forcing of late Cenozoic climate. *Nature*, 359, 117–122.
- Sambridge, M., et al., 2006. Trans-dimensional inverse problems, model comparison and the evidence. *Geophysical Journal International*, 167(2), 528–542.
- Westerhold, T., et al., 2020. An astronomically dated record of Earth's climate over the last 66 Ma. *Science*, 369(6509), 1383–1387.
- Kumar, R., et al., 2019. ArviZ: unified library for Bayesian model analysis. *JOSS*, 4(33), 1143.
- Salvatier, J., et al., 2016. Probabilistic programming in Python using PyMC3. *PeerJ CS*, 2, e55.